# Learned GNN — the GPU-tier upgrade to fixed CellGraph

Trains two learned encoders (**GraphSAGE** and **R-GCN**) on the 16,492-node cell graph and runs the
**honest head-to-head** vs the fixed SIGN propagation, on the *same* leakage-free link-prediction
benchmark (test edges removed, degree-matched hard negatives). Both beat fixed; R-GCN (per-relation
weights) is best: PPI link AUC fixed 0.826 → GraphSAGE 0.875 → **R-GCN 0.886**. Set **Runtime → GPU**.
The AUC is identical CPU vs GPU — the GPU only makes training seconds instead of minutes.
See `docs/CELLGRAPH_GNN.md`.


## 1 · Clone + install (torch)


In [ ]:
import os, sys
BR='claude/vectorize-gex-propensity-zp09w8'
if not os.path.exists('colab/cellgraph.py'):
    os.system(f'git clone -q --branch {BR} https://github.com/nikku03/cell.git')
    if os.path.isdir('cell'): os.chdir('cell')
os.system('pip -q install numpy scipy scikit-learn torch')
sys.path.insert(0,'colab'); os.makedirs('outputs/orphan', exist_ok=True)
import torch; print('device:', 'cuda' if torch.cuda.is_available() else 'cpu (set Runtime->GPU)')


## 2 · Restore the deeper model from Drive


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import glob, gzip, shutil, json
dst='outputs/orphan/cell_complete.json'
if not os.path.exists(dst):
    c=sorted(glob.glob('/content/drive/MyDrive/cell_model/**/cell_complete*.json*',recursive=True),
             key=lambda p: os.path.getsize(p), reverse=True)
    src=c[0]; print('using', src)
    (shutil.copyfileobj(gzip.open(src,'rb'),open(dst,'wb')) if src.endswith('.gz') else shutil.copy(src,dst))


## 3 · Head-to-head — learned GraphSAGE vs fixed propagation (300 epochs on GPU)


In [ ]:
import subprocess
print(subprocess.run([sys.executable,'colab/validate_cellgraph_gnn.py','300'],
      capture_output=True,text=True,env={**os.environ,'PYTHONPATH':'colab'}).stdout)


## 4 · Train a full model + inspect learned embeddings


In [ ]:
from cellgraph import load_model, node_features, build_adj
import cellgraph_gnn as gnn
D,G,name,idx = load_model(); X = node_features(G); A = build_adj(D, len(G))
r = gnn.head_to_head(D, X, A, relation='ppi', epochs=300)
print('PPI link prediction  fixed', r['fixed_auc'], ' -> learned', r['learned_auc'],
      ' (Δ', r['delta'], ',', r['device'], ')')


## Notes
- Adopt the learned encoder only where it beats the fixed bar (the validator checks this).
- Next GPU-tier items in `docs/FUTURE_IDEAS.md`: ESM-2 node features, structure-based kcat (KcatNet),
  mutant kinetics (CatPred/RealKcat).
